# Chapter 2: Pre-Model Workflow and Data Preprocessing

## Ringkasan Bab & Penjelasan Teoretis
Kualitas data masukan menentukan keberhasilan model machine learning (*garbage in, garbage out*). Bab ini membedah teknik-teknik pra-pemrosesan data menggunakan transformer dan pipeline di `scikit-learn`.

### 1. Penanganan Data Hilang (*Handling Missing Data*)
Sebagian besar algoritma machine learning tidak dapat mengeksekusi dataset yang mengandung nilai `NaN`. Tiga pendekatan imputasi utama di `scikit-learn`:
* **`SimpleImputer()`**: Mengganti nilai yang hilang dengan statistik kolom seperti `mean`, `median`, atau `most_frequent`. Ringan secara komputasi dan cocok bila data hilang $<5\%$.
* **`KNNImputer()`**: Menggunakan algoritma *k-Nearest Neighbors* untuk mengestimasi nilai hilang berdasarkan kedekatan sampel di ruang fitur. Ideal untuk fitur numerik dengan tingkat kehilangan moderat ($5\% - 10\%$).
* **`IterativeImputer()`**: Memodelkan setiap fitur yang memiliki nilai hilang sebagai fungsi regresi dari fitur-fitur lainnya secara berulang (*round-robin*), sehingga mampu menangkap hubungan multivariat.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Membuat dataset kuantitatif sintetis dengan sekitar 20% missing values
np.random.seed(2024)
n_samples = 20
n_features = 10

data = {f"Feature{i+1}": np.random.uniform(0, 100, n_samples) for i in range(n_features)}
df = pd.DataFrame(data)

for column in df.columns:
    mask = np.random.random(n_samples) < 0.2
    df.loc[mask, column] = np.nan

print("DataFrame Awal dengan Missing Values (6 baris pertama):")
display(df.head(6))

# 1. SimpleImputer
imputer = SimpleImputer(strategy="mean")
imputed_df = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)
print("Hasil SimpleImputer (Mean):")
display(imputed_df.head(6))

# 2. KNNImputer
knn_imputer = KNNImputer(n_neighbors=2)
knn_imputed_df = pd.DataFrame(knn_imputer.fit_transform(df), columns=df.columns)
print("Hasil KNNImputer (k=2):")
display(knn_imputed_df.head(6))

# 3. IterativeImputer
iterative_imputer = IterativeImputer(random_state=2024)
iterative_imputed_df = pd.DataFrame(iterative_imputer.fit_transform(df), columns=df.columns)
print("Hasil IterativeImputer:")
display(iterative_imputed_df.head(6))

### 2. Teknik Penyekalaan (*Scaling Techniques*)
Fitur dengan rentang skala yang jauh berbeda (misal usia $0-100$ vs pendapatan $0-100.000$) dapat mendominasi algoritma berbasis jarak (KNN, SVM) dan gradien.
* **Standardisasi (`StandardScaler`)**: Mengubah fitur agar memiliki rata-rata $\mu = 0$ dan simpangan baku $\sigma = 1$ (transformasi skor-Z):
  $$z = \frac{x - \mu}{\sigma}$$
* **Min-Max Scaling (`MinMaxScaler`)**: Menyekalakan ulang fitur ke dalam rentang tetap $[0, 1]$:
  $$x' = \frac{x - X_{\min}}{X_{\max} - X_{\min}}$$
* **Normalisasi Sampel (`Normalizer`)**: Menyekalakan setiap sampel individual (baris) agar memiliki norma unit ($L_1$ atau $L_2$), di mana $\|x\|_2 = \sqrt{\sum_{i=1}^{n} x_i^2}$.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer

# 1. StandardScaler
scaler = StandardScaler()
scaled_df = pd.DataFrame(scaler.fit_transform(iterative_imputed_df), columns=iterative_imputed_df.columns)
print("Hasil StandardScaler:")
display(scaled_df.head(6))

# 2. MinMaxScaler
minmax_scaler = MinMaxScaler()
minmax_scaled_df = pd.DataFrame(minmax_scaler.fit_transform(iterative_imputed_df), columns=iterative_imputed_df.columns)
print("Hasil MinMaxScaler:")
display(minmax_scaled_df.head(6))

# 3. Normalizer
normalizer = Normalizer()
normalized_df = pd.DataFrame(normalizer.fit_transform(iterative_imputed_df), columns=iterative_imputed_df.columns)
print("Hasil Normalizer:")
display(normalized_df.head(6))

### 3. Encoding Variabel Kategorikal & `ColumnTransformer`
* **`OneHotEncoder()`**: Mengubah variabel kategorikal nominal menjadi kolom-kolom biner terpisah untuk setiap kategori.
* **`LabelEncoder()`**: Memberikan bilangan bulat unik ($0, 1, 2, \dots$) untuk setiap kategori (perlu berhati-hati karena dapat menimbulkan kesan urutan ordinal pada fitur nominal).
* **`ColumnTransformer()`**: Menerapkan transformasi berbeda secara paralel pada kolom numerik dan kolom kategorikal di dalam satu dataset campuran.

In [ ]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer

np.random.seed(2024)
categories = ["A", "B", "C", "D"]
categorical_data = pd.DataFrame({
    "Department": np.random.choice(categories, size=20),
    "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
    "Location": np.random.choice(["NY", "SF", "LA", "CHI"], size=20),
})

# 1. OneHotEncoder
onehot_encoder = OneHotEncoder(sparse_output=False)
onehot_encoded_data = onehot_encoder.fit_transform(categorical_data)
onehot_encoded_df = pd.DataFrame(onehot_encoded_data, columns=onehot_encoder.get_feature_names_out())
print("Hasil OneHotEncoder:")
display(onehot_encoded_df.head(6))

# 2. LabelEncoder
label_encoder = LabelEncoder()
label_encoded_df = pd.DataFrame()
for column in categorical_data.columns:
    label_encoded_df[f"{column}_encoded"] = label_encoder.fit_transform(categorical_data[column])
print("Hasil LabelEncoder:")
display(label_encoded_df.head(6))

# 3. ColumnTransformer untuk Dataset Campuran
np.random.seed(2024)
mixed_data = pd.DataFrame({
    "Age": np.random.randint(25, 65, size=20),
    "Salary": np.round(np.random.normal(60000, 15000, size=20), 2),
    "Experience": np.random.randint(1, 20, size=20),
    "Department": np.random.choice(["IT", "HR", "Sales", "Finance"], size=20),
    "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
})

column_transformer = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["Age", "Salary", "Experience"]),
        ("cat", OneHotEncoder(sparse_output=False), ["Department", "Position"]),
    ],
    remainder="passthrough",
)

transformed_data = column_transformer.fit_transform(mixed_data)
numeric_cols = ["Age_scaled", "Salary_scaled", "Experience_scaled"]
categorical_cols = column_transformer.named_transformers_["cat"].get_feature_names_out(["Department", "Position"])
transformed_df = pd.DataFrame(transformed_data, columns=numeric_cols + list(categorical_cols))
print("Hasil ColumnTransformer:")
display(transformed_df.head(6))

### 4. Pipelines, Feature Engineering, & Latihan Praktis (California Housing)
* **Pipelines**: Merangkai tahapan imputasi, penyekalaan, dan pemodelan secara berurutan setelah `train_test_split` guna mencegah *data leakage*.
* **Feature Extraction**: Menciptakan fitur baru menggunakan `PolynomialFeatures()` (kombinasi polinomial dan interaksi) serta `KBinsDiscretizer()` (pengelompokan nilai kontinu ke dalam interval diskrit).
* **Feature Selection**: Memilih fitur paling relevan menggunakan *Recursive Feature Elimination* (`RFE`) dan `SelectFromModel()`.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn import set_config
from sklearn.preprocessing import PolynomialFeatures, KBinsDiscretizer
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.linear_model import LinearRegression
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_squared_error, r2_score

# Pemisahan fitur dan target sebelum masuk ke Pipeline
X = transformed_df.iloc[:, :-1]
y = transformed_df.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2024)

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
])

X_train_transformed = pd.DataFrame(pipeline.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_transformed = pd.DataFrame(pipeline.transform(X_test), columns=X_test.columns, index=X_test.index)

set_config(display="diagram")
display(pipeline)

# PolynomialFeatures & KBinsDiscretizer
poly = PolynomialFeatures(degree=2)
poly_features_df = pd.DataFrame(
    poly.fit_transform(X_train_transformed),
    columns=poly.get_feature_names_out(X_train_transformed.columns)
)

kbins = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="uniform")
binned_df = pd.DataFrame(kbins.fit_transform(X_train_transformed), columns=X_train_transformed.columns)

# RFE & SelectFromModel
rfe = RFE(estimator=LinearRegression(), n_features_to_select=1)
rfe.fit(X_train_transformed, y_train)
print("RFE Feature Ranking:", rfe.ranking_)

selector = SelectFromModel(estimator=LinearRegression(), prefit=False, threshold="mean")
selector.fit(X_train_transformed, y_train)
feature_importance = pd.DataFrame({
    "Feature": X_train_transformed.columns,
    "Importance": selector.estimator_.coef_,
    "Selected": selector.get_support()
})
display(feature_importance.sort_values("Importance", key=abs, ascending=False))

# Practical Exercise: Comprehensive Pipeline pada California Housing Dataset
housing = fetch_california_housing()
X_h_train, X_h_test, y_h_train, y_h_test = train_test_split(
    housing.data, housing.target, test_size=0.2, random_state=2024
)

comp_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("selector", SelectFromModel(LinearRegression(), threshold="mean")),
    ("model", LinearRegression())
])
comp_pipeline.fit(X_h_train, y_h_train)
y_h_pred = comp_pipeline.predict(X_h_test)
print(f"California Housing Pipeline -> MSE: {mean_squared_error(y_h_test, y_h_pred):.4f}, R2: {r2_score(y_h_test, y_h_pred):.4f}")